In [9]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.13.0+cu130
13.0
True
NVIDIA GeForce RTX 3060 Laptop GPU


In [10]:
import json
import random
import numpy as np

In [11]:
with open("Fin-RATE/qa/LT-QA.json", "r",encoding="utf-8") as file:
    ltqa = json.load(file)

In [12]:
nqa = 100
smoke_test_ltqa = random.sample(ltqa,nqa)
sample_docs = set()
for sample_qa in smoke_test_ltqa:
    docs = set(sample_qa["doc_ids"])
    sample_docs.update(docs)

#print(sample_docs)
print(len(sample_docs))

233


In [13]:
len(ltqa)

2500

In [14]:
#System 1: Normal Vector Retrieval
from vector_retrieval import retrieval_pipeline
recalls = []
mrr = 0
for qa in smoke_test_ltqa:
    chunks = retrieval_pipeline(qa["question"])
    golden = qa["doc_ids"]
    #Recall 5
    recall5 = len(set(chunks[:5]) & set(golden))/len(golden)
    recall10 = len(set(chunks[:10]) & set(golden))/len(golden)

    recalls.append((recall5,recall10))

    rr = 0
    for rank, item in enumerate(chunks, start=1):
        if item in golden:
            rr = 1 / rank
            break
    mrr += rr

print(np.mean([recs[0] for recs in recalls]))
print(np.mean([recs[1] for recs in recalls]))
#print(mrr/nqa)


0.049166666666666664
0.08


In [ ]:
#System 2: ChromaDB Vector Retrieval
from ChromaSetup import create_embedding_function, build_chroma_database, retrieve_relevant_chunks



{'corpus_path': 'C:\\Users\\hello2\\Documents\\Inception\\Fin-RATE\\corpus\\corpus\\corpus.jsonl',
 'database_dir': 'C:\\Users\\hello2\\Documents\\Inception\\Fin-RATE\\chroma_db',
 'collection_name': 'fin_rate',
 'documents_seen': 15311,
 'chunks_indexed': 78203,
 'collection_count': 78203,
 'chunk_size_tokens': 512,
 'chunk_overlap_tokens': 64,
 'embedding_function': 'SentenceTransformerEmbeddingFunction',
 'build_started_at_utc': '2026-07-29T06:28:26.672182+00:00',
 'build_finished_at_utc': '2026-07-29T06:36:43.610730+00:00',
 'build_seconds': 496.93839070003014}

In [1]:
!python ChromaSetup.py --build --embedding-backend bge --device cuda

^C


In [ ]:
recalls = []
for qa in smoke_test_ltqa:
    chunks = retrieve_relevant_chunks(qa["question"])
    golden = qa["doc_ids"]
    #Recall 5
    recall5 = len(set([chun['metadata']['doc_id'] for chun in chunks][:5]) & set(golden))/len(golden)
    recall10 = len(set([chun['metadata']['doc_id'] for chun in chunks][:10]) & set(golden))/len(golden)
    recalls.append((recall5,recall10))

print(np.mean([recs[0] for recs in recalls]))
print(np.mean([recs[1] for recs in recalls]))

In [17]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("BAAI/bge-m3", device=device)

embeddings = model.encode(
    ["What is reranking in RAG?"],
    batch_size=8,
    normalize_embeddings=True,
)
print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\hello2\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hello2\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

(1, 1024)


In [16]:
#System 3: CRAG 
'''

from ChromaSetup import create_embedding_function, build_chroma_database, retrieve_relevant_chunks

ef = create_embedding_function(
    backend="sentence-transformers",
    device="cuda",
)

recalls = []
for qa in smoke_test_ltqa:
    chunks = retrieve_relevant_chunks(qa["question"])
    golden = qa["doc_ids"]
    #Recall 5
    recall5 = len(set([chun['metadata']['doc_id'] for chun in chunks][:5]) & set(golden))/len(golden)
    recall10 = len(set([chun['metadata']['doc_id'] for chun in chunks][:10]) & set(golden))/len(golden)
    recalls.append((recall5,recall10))

print(np.mean([recs[0] for recs in recalls]))
print(np.mean([recs[1] for recs in recalls]))

'''

'\n\nfrom ChromaSetup import create_embedding_function, build_chroma_database, retrieve_relevant_chunks\n\nef = create_embedding_function(\n    backend="sentence-transformers",\n    device="cuda",\n)\n\nrecalls = []\nfor qa in smoke_test_ltqa:\n    chunks = retrieve_relevant_chunks(qa["question"])\n    golden = qa["doc_ids"]\n    #Recall 5\n    recall5 = len(set([chun[\'metadata\'][\'doc_id\'] for chun in chunks][:5]) & set(golden))/len(golden)\n    recall10 = len(set([chun[\'metadata\'][\'doc_id\'] for chun in chunks][:10]) & set(golden))/len(golden)\n    recalls.append((recall5,recall10))\n\nprint(np.mean([recs[0] for recs in recalls]))\nprint(np.mean([recs[1] for recs in recalls]))\n\n'